# 🔴 Notebook 1 — Pipeline de Clasificación + Explicación con LLM
> **El modelo predice, el LLM explica**

En este notebook vamos a:
1. Entrenar un clasificador de churn con `scikit-learn`
2. Pasar las predicciones + features al LLM (Gemini via LangChain)
3. Generar explicaciones en lenguaje natural para cada predicción

**Módulo:** IA Generativa — Clase 4: Pipelines ML + GenAI

## 📖 ¿Qué hace este notebook? — Léelo antes de tocar nada

Imagina que tienes una lista de miles de clientes y quieres saber **quién va a volver a comprar y quién no**.

Este notebook hace dos cosas en orden:

1. **Un modelo matemático** (Random Forest) mira los datos de cada cliente y calcula la probabilidad de que vuelva a comprar. Esto lo hace rápido, con miles de clientes a la vez.
2. **Una inteligencia artificial** (Gemini, el mismo tipo de IA que ChatGPT pero de Google) lee esa probabilidad y escribe un mensaje de marketing personalizado para ese cliente en lenguaje humano.

**El resultado final:** para cada cliente, un texto listo para enviarle — ya sea un mensaje VIP, una oferta o una campaña de recuperación.

---
> 💡 **Regla de oro de este notebook:** el código no hay que entenderlo al 100%. Lo importante es entender **qué entra, qué sale y por qué**.


## 1. Instalación de dependencias

### 🔧 ¿Qué es "instalar dependencias"?

Python por sí solo no sabe hacer Machine Learning ni hablar con Gemini.  
Necesita que le instalemos **herramientas adicionales** (llamadas librerías o paquetes).

Es exactamente igual que instalar una app en el móvil: la descargas una vez y ya puedes usarla.

Las que instalamos aquí son:

| Librería | Para qué sirve |
|---|---|
| `pandas` | Leer y manejar tablas de datos (como Excel pero en código) |
| `numpy` | Hacer cálculos matemáticos rápidos |
| `scikit-learn` | Construir el modelo que predice si el cliente va a recomprar |
| `langchain` | El "conector" que nos permite hablar con Gemini desde Python |
| `langchain-google-genai` | La parte específica de LangChain para usar la IA de Google |
| `python-dotenv` | Leer contraseñas guardadas en un archivo seguro (el .env) |
| `transformers`, `torch` | Librerías de IA avanzada (instaladas por precaución, aunque no se usen directamente aquí) |


#### ▶️ Código: comprobar el entorno de Python

Esta celda simplemente nos dice **en qué "carpeta virtual" de Python estamos trabajando**.  
Es como comprobar que estás en el despacho correcto antes de ponerte a trabajar.  
No afecta al resultado final — es solo una comprobación técnica.


In [1]:
import sys
print(sys.prefix)

c:\Users\Oscar\OneDrive - FM4\Escritorio\EVOLVE\Data Science\EVOLVE\Victor_Prior_IA_Generativa\Proyecto_Modulo_IAGen\.venv


#### ▶️ Código: instalar las librerías principales

`pip install` = "instala esto en mi Python".  
La `-q` al final significa "hazlo en silencio, sin mostrar tanto texto".  
Solo hay que ejecutar esto **una vez**. Si ya están instaladas, no pasa nada.


In [2]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q 


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### ▶️ Código: instalar librerías de IA adicionales

Igual que el anterior pero para las librerías de inteligencia artificial más avanzadas.  
`transformers` y `torch` son la base sobre la que funcionan muchos modelos de IA modernos.


In [3]:
!pip install -q transformers accelerate torch


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuración de API Key

### 🔑 ¿Qué es una API Key?

Una API Key es como **una contraseña de un solo uso que identifica quién está usando el servicio**.

Para usar Gemini (la IA de Google) desde nuestro código, Google necesita saber que somos nosotros y no cualquiera.  
Esa "identificación" es la API Key.

**¿Por qué está en un archivo `.env` y no escrita directamente en el código?**  
Porque si la escribieras en el código y compartieras el notebook, **cualquiera podría usar tu cuenta de Google y generarte costes**.  
El archivo `.env` es como una caja fuerte separada: el código la abre, coge la contraseña, pero la contraseña en sí nunca es visible en el notebook.

> ⚠️ Nunca compartas tu API Key con nadie ni la escribas directamente en el código.


#### ▶️ Código: cargar la API Key desde el archivo seguro

- `load_dotenv(...)` — abre el archivo `.env` que está en la ruta indicada y carga las variables guardadas en él.
- `os.getenv("Gemini_API_Key_001")` — coge del archivo la variable que se llama `Gemini_API_Key_001` y la guarda en `GOOGLE_API_KEY`.
- Las líneas entre `'''` son **código comentado** (desactivado). Son una forma alternativa de cargar la key si usáramos Google Colab en vez de un ordenador local. No se ejecutan.
- Al final imprime ✅ para confirmar que todo fue bien.


In [2]:
import os
from dotenv import load_dotenv
#from google.colab import userdata

load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")

# Opción A: desde .env
GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")

'''# Opción B: directamente (solo para desarrollo)
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')'''

print("✅ API Key cargada")

✅ API Key cargada


## 3. Cargar dataset

### 📊 ¿Qué es el dataset y qué contiene?

El dataset es el archivo `df_rfm_final.csv` — una tabla con datos de clientes reales.

Cada fila es un cliente. Las columnas más importantes son:

| Columna | Qué significa |
|---|---|
| `customer_id` | El identificador único de cada cliente |
| `recency` | Hace cuántos días compró por última vez |
| `numero_de_compras` | Cuántas veces ha comprado en total |
| `monetary` | Cuánto dinero ha gastado en total (€) |
| `nombre_cluster` | A qué segmento de cliente pertenece (ej: VIP, en riesgo, etc.) |
| `prob_recompra` | La probabilidad (entre 0 y 1) de que vuelva a comprar |

**RFM** es una técnica de marketing clásica que analiza a los clientes por:
- **R**ecency (Recencia): cuándo compraron por última vez
- **F**requency (Frecuencia): cuántas veces han comprado
- **M**onetary (Monetario): cuánto han gastado


#### ▶️ Código: importar herramientas y cargar los datos

**Importaciones:**
- `pandas as pd` — la librería para manejar tablas. La abreviamos como `pd` para no escribir el nombre completo cada vez.
- `numpy as np` — para cálculos numéricos. Abreviado como `np`.
- `RandomForestClassifier` — el algoritmo que predice si el cliente va a recomprar. (Un "bosque aleatorio" es un conjunto de muchos árboles de decisión que votan juntos.)
- `train_test_split` — divide los datos en dos grupos: uno para entrenar el modelo y otro para comprobar si acierta.
- `classification_report` — genera un informe de cuánto acierta el modelo.
- `LabelEncoder` — convierte texto en números (porque los modelos matemáticos solo entienden números).

**Las 3 últimas líneas:**
- `pd.read_csv(...)` — lee el archivo CSV y lo convierte en una tabla que Python puede manejar. La guarda en la variable `df`.
- `df.shape` — nos dice cuántas filas (clientes) y columnas tiene la tabla.
- `df.head()` — muestra los 5 primeros clientes para ver cómo son los datos.


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/df_rfm_final.csv')
print(f"Dataset: {df.shape}")
print(f"Recompra rate: {df['prob_recompra'].mean():.1%}")
df.head()

Dataset: (4947, 14)
Recompra rate: 53.1%


,customer_id,ultima_compra,primera_compra,numero_de_compras,monetary,recency,dias_entre_pedidos,tasa_devolucion,invoice,target,cluster,prob_recompra,segmento_prob,nombre_cluster
0,12347,2011-04-07 10:43:00,2010-10-31 14:20:00,4,2434.96,63,52.333333,0.0,4,1,2,0.917191,alta,Desconectados
1,12348,2011-04-05 10:47:00,2010-09-27 14:59:00,4,1709.40,65,63.000000,0.0,4,1,2,0.868305,alta,Desconectados
2,12349,2010-10-28 08:23:00,2009-12-04 12:49:00,4,2646.99,224,109.000000,0.2,4,1,2,0.950565,alta,Desconectados
3,12350,2011-02-02 16:01:00,2011-02-02 16:01:00,1,334.40,126,0.000000,0.0,1,0,0,0.135661,baja,Diamante
4,12351,2010-11-29 15:23:00,2010-11-29 15:23:00,1,300.93,191,0.000000,0.0,1,0,0,0.344013,media,Diamante


## 4. Configurar LangChain + Gemini

### 🤖 ¿Qué es LangChain y qué tiene que ver con Gemini?

**Gemini** es el modelo de inteligencia artificial de Google (como ChatGPT pero de Google).  
**LangChain** es una librería de Python que actúa de **intermediario**: facilita que nuestro código hable con Gemini de forma ordenada.

En esta sección preparamos conceptualmente las tres piezas que luego el código de la sección 5 monta:

| Pieza | Qué es | Cuándo se construye |
|---|---|---|
| **Prompt** | Las instrucciones que le damos a Gemini para cada cliente | Paso 3 del código |
| **Modelo (llm)** | La conexión real con Gemini — el que responde | Paso 4 del código |
| **Cadena (chain)** | Une el prompt + modelo + salida en una sola tubería | Paso 5 del código |

> ⚠️ En esta sección **no hay celda de código**. Todo el montaje ocurre en la sección 5 en un único bloque. Aquí solo entendemos qué se va a construir y por qué.


#### 📋 Lo que esta sección aporta al código: el Prompt (paso 3)

El **prompt** es la pieza más importante de entender antes de ver el código.  
Es el mensaje que nuestro código le manda a Gemini para cada cliente. Funciona como una plantilla con huecos:

```
Eres un experto en Growth Marketing. Analiza al siguiente cliente:
- Recency: {recency} días           ← se rellena con el dato real del cliente
- Frecuencia: {frecuencia} compras  ← se rellena con el dato real del cliente
- Probabilidad de Recompra: {probabilidad}
...
```

Cada vez que el código procesa un cliente, rellena esos huecos `{}` con sus datos reales  
y le manda ese mensaje completo a Gemini. Gemini lo lee y escribe el mensaje de marketing.

Las instrucciones al final del prompt le dicen a Gemini qué tipo de respuesta dar según la probabilidad:
- **> 70%** → mensaje VIP exclusivo  
- **30–70%** → oferta o cupón  
- **< 30%** → descuento agresivo para recuperar al cliente


## 5. Generar explicaciones para los primeros casos

### 🔍 ¿Qué hace el código de esta sección?

Esta sección contiene **una sola celda de código que hace 6 cosas en orden**.  
Es donde ocurre todo: la conexión con Google, la selección del modelo, la construcción de la cadena y el resultado.

Los 6 pasos son:

| Paso | Qué hace |
|---|---|
| **1** | Conecta la librería de Google con tu API Key |
| **2** | Pregunta a Google qué modelos tienes disponibles y los lista |
| **3** | Construye el prompt (la plantilla de instrucciones para Gemini) |
| **4** | Prueba los modelos uno a uno hasta que uno responda |
| **5** | Con el modelo que funcionó, monta la cadena completa (`chain`) |
| **6** | Confirma qué modelo quedó activo o avisa si ninguno funcionó |

Al terminar esta celda, `chain` está lista para usarse en el resto del notebook.


#### ▶️ Pasos 1 y 2 — Conectar con Google y listar modelos disponibles

**Paso 1 — `genai.configure(api_key=GOOGLE_API_KEY)`**  
Le dice a la librería de Google quién somos. A partir de aquí puede hablar con los servidores de Google en nuestro nombre.

**Paso 2 — construir `modelos_disponibles`**  
Pide a Google la lista completa de modelos y filtra solo los que:
- Soportan generación de texto (`generateContent`)
- Son modelos Gemini (descarta otros tipos como los de análisis de imágenes o embeddings)

El resultado es una lista de IDs como `gemini-2.5-flash`, `gemini-1.5-pro`, etc.  
Esos son los candidatos que se probarán en el paso 4.


#### ▶️ Pasos 3, 4, 5 y 6 — Construir la cadena con el modelo que funcione

**Paso 3 — `prompt = ChatPromptTemplate.from_template(...)`**  
Construye la plantilla de instrucciones que vimos en la sección 4.  
Se hace aquí (antes del bucle) porque el prompt es el mismo para todos los modelos.

**Paso 4 — el bucle `for modelo in modelos_disponibles`**  
Recorre la lista de modelos y para cada uno:
- Crea una conexión temporal (`llm_temp`)
- Le manda un mensaje de una palabra para ver si responde (`"Responde solo con la palabra: OK"`)
- Si responde → pasamos al paso 5
- Si falla (saturado, sin cuota, no disponible) → imprime ❌ y prueba el siguiente

**Paso 5 — `chain = prompt | llm_temp | StrOutputParser()`**  
Solo se ejecuta cuando un modelo respondió bien.  
Une las tres piezas: el prompt + el modelo que funciona + el parser de salida (que convierte la respuesta en texto legible).  
El `|` funciona como una tubería: la salida de cada pieza entra en la siguiente.

**Paso 6 — resultado final**  
Si `chain` no es `None` (es decir, algún modelo funcionó) → imprime ✅ con el nombre del modelo activo.  
Si `chain` sigue siendo `None` → imprime 🚨 y hay que revisar la API Key o la conexión.


In [ ]:
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Configurar la API Key (usando la que ya cargaste de tu .env)
genai.configure(api_key=GOOGLE_API_KEY)

# 2. Listar modelos que soportan generación de contenido
modelos_disponibles = [
    m.name.split("/")[-1]          # nos quedamos solo con el ID corto, ej: "gemini-2.5-flash"
    for m in genai.list_models()
    if "generateContent" in m.supported_generation_methods
    and "gemini" in m.name         # filtramos solo modelos Gemini (descartamos embeddings, etc.)
]

print(f"🔍 Modelos encontrados para probar: {len(modelos_disponibles)}")
for m in modelos_disponibles:
    print(f"   - {m}")

# 3. Configuramos el prompt para construir el chain
prompt = ChatPromptTemplate.from_template("""
Eres un experto en Growth Marketing. Analiza al siguiente cliente:

**Datos RFM:**
- Recency: {recency} días
- Frecuencia: {frecuencia} compras
- Valor Monetario: {monetary}€
- Segmento: {nombre_cluster}

**Métrica de Predicción:**
- Probabilidad de Recompra: {probabilidad:.1%} 

Instrucciones:
1. Si la probabilidad es > 70%, redacta un mensaje de agradecimiento VIP muy exclusivo.
2. Si está entre 30% y 70%, ofrece un incentivo basado en su historial (cupón o producto).
3. Si es < 30%, genera una estrategia de "Win-back" con un descuento agresivo para evitar que se vaya.

Escribe el mensaje directamente al cliente en un tono profesional y cercano.
""")

# 4. Configurar el modelo probando uno a uno hasta que uno funcione
chain = None           # aquí guardaremos la chain cuando un modelo responda bien
modelo_activo = None   # aquí guardaremos el nombre del modelo que funcionó

for modelo in modelos_disponibles:
    print(f"\n⏳ Probando modelo: {modelo} ...")
    try:
        # Crear conexión temporal con este modelo
        llm_temp = ChatGoogleGenerativeAI(
            model=modelo,
            google_api_key=GOOGLE_API_KEY,
            temperature=0.7
        )

        # Mandar un mensaje de prueba mínimo para ver si responde
        # (usamos invoke directo, sin el prompt completo, para que sea rápido)
        test = llm_temp.invoke("Responde solo con la palabra: OK")

        # Si llega aquí sin error, el modelo está disponible → construimos la chain real
# 5. Crear la cadena (Chain)
        chain = prompt | llm_temp | StrOutputParser()
        modelo_activo = modelo
        print(f"✅ Modelo disponible y funcionando: {modelo}")
        break  # salimos del bucle, ya no necesitamos probar más

    except Exception as e:
        # El modelo falló (saturado, no disponible, error de cuota, etc.)
        print(f"❌ {modelo} no disponible → {str(e)[:120]}")  # mostramos solo los primeros 120 caracteres del error

# 6. Resultado final
if chain is not None:
    print(f"\n🚀 Chain configurada con: {modelo_activo}")
    print("   Puedes seguir con la sección 5 y la sección 7 sin cambiar nada más.")
else:
    print("\n🚨 Ningún modelo respondió. Comprueba tu API Key o tu conexión a internet.")


🔍 Modelos encontrados para probar: 24
   - gemini-2.5-flash
   - gemini-2.5-pro
   - gemini-2.0-flash
   - gemini-2.0-flash-001
   - gemini-2.0-flash-lite-001
   - gemini-2.0-flash-lite
   - gemini-2.5-flash-preview-tts
   - gemini-2.5-pro-preview-tts
   - gemini-flash-latest
   - gemini-flash-lite-latest
   - gemini-pro-latest
   - gemini-2.5-flash-lite
   - gemini-2.5-flash-image
   - gemini-3-pro-preview
   - gemini-3-flash-preview
   - gemini-3.1-pro-preview
   - gemini-3.1-pro-preview-customtools
   - gemini-3.1-flash-lite-preview
   - gemini-3-pro-image-preview
   - gemini-3.1-flash-image-preview
   - gemini-3.1-flash-tts-preview
   - gemini-robotics-er-1.5-preview
   - gemini-robotics-er-1.6-preview
   - gemini-2.5-computer-use-preview-10-2025

⏳ Probando modelo: gemini-2.5-flash ...
❌ gemini-2.5-flash no disponible → Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 

⏳ Probando modelo: gemini-2.5-pro ...
❌ g

#### ▶️ Código: generar el mensaje de marketing para los 3 primeros clientes

Con `chain` ya configurada por el código anterior, esta celda la usa para producir resultados reales.

- `df.head(3)` — coge los 3 primeros clientes de la tabla.
- `for i, (idx, row) in enumerate(casos_prueba.iterrows())` — recorre los 3 clientes uno a uno. Cada vez, `row` contiene todos los datos de ese cliente.
- Las líneas `print(f"...")` — imprimen una separación visual con el ID y el segmento del cliente.
- `chain.invoke({...})` — manda los datos del cliente a Gemini a través de la cadena. Gemini lee el prompt con los huecos rellenados y devuelve el texto del mensaje.
  - `int(row["recency"])` — días desde la última compra, como número entero.
  - `int(row["numero_de_compras"])` — número de compras. En el CSV se llama `numero_de_compras`, en el prompt lo llamamos `frecuencia`.
  - `float(row["monetary"])` — euros gastados, como número decimal.
- `print(resultado)` — imprime el texto que escribió Gemini para ese cliente.

**Al ejecutar esta celda verás 3 mensajes de marketing distintos**, uno por cliente, adaptados a su probabilidad de recompra.


In [7]:
# 1. Selección de ejemplos (usamos los 3 primeros de dr_rfm_final)
casos_prueba = df.head(3) 

for i, (idx, row) in enumerate(casos_prueba.iterrows()):
    print(f"\n{'='*60}")
    print(f"🚀 ESTRATEGIA PARA CLIENTE ID: {row['customer_id']}")
    print(f"📊 Cluster: {row['nombre_cluster']} | Prob. Recompra: {row['prob_recompra']:.1%}")
    print(f"{'='*60}")

    # Mapeo exacto entre tus columnas y lo que espera el Prompt
    # Nota: Usamos 'numero_de_compras' para alimentar la 'frecuencia' del LLM
    resultado = chain.invoke({
        "recency": int(row["recency"]),
        "frecuencia": int(row["numero_de_compras"]),
        "monetary": float(row["monetary"]),
        "nombre_cluster": row["nombre_cluster"],
        "probabilidad": float(row["prob_recompra"])
    })

    print(resultado)


🚀 ESTRATEGIA PARA CLIENTE ID: 12347
📊 Cluster: Desconectados | Prob. Recompra: 91.7%
Como experto en Growth Marketing, he analizado el perfil de este cliente. A pesar de estar etiquetado como "Desconectado" por sus 63 días de inactividad, su **Probabilidad de Recompra del 91.7%** y su **Ticket Medio de ~608€** lo convierten en un activo de altísimo valor (High-Value Customer) que simplemente necesita un "empujón" de reconocimiento para reactivarse.

Dado que la probabilidad es superior al 70%, aplicaré la **Estrategia de Fidelización VIP Exclusiva**, centrada en el sentimiento de pertenencia y el estatus, más que en el descuento directo.

Aquí tienes la propuesta de comunicación:

***

**Asunto: Una distinción especial para ti, [Nombre del Cliente]**

Hola, **[Nombre del Cliente]**:

Es un placer saludarte de nuevo. 

En **[Nombre de tu Marca]**, revisamos periódicamente la trayectoria de las personas que confían en nosotros y tu perfil destaca de manera especial. Con cuatro compras s

## 7. Pipeline completo como función reutilizable

### 🔁 ¿Qué es una función reutilizable y para qué sirve aquí?

La celda de la sección 5 que genera mensajes para 3 clientes está escrita de forma "suelta":  
si mañana quieres procesar 200 clientes, tendrías que modificar el código.

Una **función** lo resuelve. Es como crear un botón personalizado:  
le das el dataframe y cuántos clientes procesar, y él solo hace el trabajo.

La función `generar_estrategias_rfm` hace exactamente lo mismo que la sección 5 pero:
- Acepta cualquier número de clientes (o todos si no indicas nada)
- Guarda los resultados en una tabla nueva con una columna `mensaje_gemini`
- Imprime un resumen al final con distribución por segmento y probabilidad media

> 💡 Para usarla solo necesitas que `chain` y `df` estén cargados (secciones 3 y 5 ejecutadas).


In [ ]:
# ─────────────────────────────────────────────────────────────
# FUNCIÓN REUTILIZABLE: generar_estrategias_rfm
#
# REQUISITOS PREVIOS: haber ejecutado las secciones 3 y 5
#   - df    → tabla de clientes cargada en sección 3
#   - chain → cadena LangChain configurada en sección 5
# ─────────────────────────────────────────────────────────────

def generar_estrategias_rfm(df, chain, n_clientes=None):
    """
    Parámetros:
        df          → la tabla de clientes (el dataframe completo)
        chain       → la cadena LangChain ya configurada con el prompt y Gemini
        n_clientes  → cuántos clientes procesar. Si no pones nada, procesa TODOS.

    Devuelve:
        Un dataframe con las columnas originales + una columna nueva 'mensaje_gemini'
    """

    # Si no se especifica número, procesamos todos los clientes del dataframe
    subset = df if n_clientes is None else df.head(n_clientes)

    resultados = []  # lista donde iremos acumulando los resultados de cada cliente

    for i, (idx, row) in enumerate(subset.iterrows()):

        # Separador visual por cliente
        print(f"\n{'='*60}")
        print(f"🚀 ESTRATEGIA PARA CLIENTE ID: {row['customer_id']}")
        print(f"📊 Cluster: {row['nombre_cluster']} | Prob. Recompra: {row['prob_recompra']:.1%}")
        print(f"{'='*60}")

        # Llamada a Gemini — rellena los huecos del prompt con los datos reales del cliente
        # y devuelve el texto generado
        mensaje = chain.invoke({
            "recency"        : int(row["recency"]),            # días desde última compra
            "frecuencia"     : int(row["numero_de_compras"]),  # nº total de compras
            "monetary"       : float(row["monetary"]),         # euros gastados
            "nombre_cluster" : row["nombre_cluster"],          # segmento del cliente
            "probabilidad"   : float(row["prob_recompra"])     # probabilidad de recompra (0 a 1)
        })

        print(mensaje)  # muestra el mensaje generado por Gemini en pantalla

        # Guardamos los datos del cliente + el mensaje en la lista de resultados
        resultados.append({
            "customer_id"      : row["customer_id"],
            "nombre_cluster"   : row["nombre_cluster"],
            "prob_recompra"    : row["prob_recompra"],
            "recency"          : row["recency"],
            "numero_de_compras": row["numero_de_compras"],
            "monetary"         : row["monetary"],
            "mensaje_gemini"   : mensaje   # columna nueva con el texto generado
        })

    # Convertimos la lista en una tabla (dataframe) para poder exportarla o analizarla
    df_resultados = pd.DataFrame(resultados)

    # Resumen final
    print(f"\n{'='*60}")
    print(f"✅ Proceso completado — {len(df_resultados)} clientes procesados")
    print(f"\n📊 Distribución por segmento:")
    print(df_resultados["nombre_cluster"].value_counts().to_string())
    print(f"\n📈 Probabilidad media de recompra: {df_resultados['prob_recompra'].mean():.1%}")
    print(f"{'='*60}")

    return df_resultados  # devuelve la tabla completa con todos los mensajes

df_con_mensajes = generar_estrategias_rfm(df, chain, n_clientes=3)

# Vista previa de la tabla resultado
df_con_mensajes[["customer_id", "nombre_cluster", "prob_recompra", "mensaje_gemini"]].head()


In [ ]:
# ─────────────────────────────────────────────────────────────
# CÓMO USAR LA FUNCIÓN
# 
# # PASO 1 — Asegúrate de haber ejecutado estas secciones antes:
#   ✅ Sección 1: instalación de librerías
#   ✅ Sección 2: carga de la API Key  →  GOOGLE_API_KEY disponible
#   ✅ Sección 3: carga del dataset    →  df disponible
#   ✅ Sección 5: selección de modelo  →  chain disponible
#
# Si alguna de esas celdas no está ejecutada, esta función fallará.
# ─────────────────────────────────────────────────────────────
# PASO 2 — Elige cuántos clientes procesar y ejecuta
#   n_clientes=5   → procesa los 5 primeros
#   n_clientes=100 → procesa los 100 primeros
#   sin parámetro  → procesa todos
#
# Opción A: procesar un número concreto de clientes
df_con_mensajes = generar_estrategias_rfm(df, chain, n_clientes=3)

# Opción B: procesar TODOS los clientes del dataset
# df_con_mensajes = generar_estrategias_rfm(df, chain)

# ─────────────────────────────────────────────────────────────
# PASO 3 — Ver la tabla con los mensajes generados
df_con_mensajes[["customer_id", "nombre_cluster", "prob_recompra", "mensaje_gemini"]]

# PASO 4 (opcional) — Exportar los resultados a un CSV
# df_con_mensajes.to_csv("estrategias_clientes.csv", index=False)
# ─────────────────────────────────────────────────────────────

## 🎯 Conclusiones

- El modelo ML es **rápido y escalable** para hacer predicciones masivas
- El LLM agrega una **capa de interpretabilidad** en lenguaje natural
- Este patrón es muy útil en producción para **equipos no técnicos** que necesitan actuar sobre predicciones

**Patrón clave:** `ML Model → Predicción + Features → LLM → Explicación accionable`